In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()


StatementMeta(, 94260921-8355-492b-9249-0421733c78fb, 3, Finished, Available, Finished, False)

### **Reading All Bronze Files**

In [2]:
df1 = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_customers")
df2 = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_geolocation")
df3 = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_order_items")
df4 = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_order_payments")
df5 = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_order_reviews")
df6 = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_orders")
df7 = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_products")
df8 = spark.read.table("Olist_Bronze_Lakehouse.dbo.olist_sellers")
df9 = spark.read.table("Olist_Bronze_Lakehouse.dbo.product_category_name_translation")


df10 = spark.read.table("Olist_Silver_Lakehouse.dbo.ref_country_state")
df11 = spark.read.table("Olist_Silver_Lakehouse.dbo.ref_city_master")

StatementMeta(, 94260921-8355-492b-9249-0421733c78fb, 4, Finished, Available, Finished, False)

### **Table : olist_customers**

In [42]:
display(df1)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 44, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 11b88e65-02e7-49cf-8798-75d42fef2f2c)

##### **Check for null**

In [43]:
df1_nulls_check = (
    df1.filter(col("customer_id").isNull())
)

df1_nulls_check2 = (
    df1.filter(
        (col("customer_zip_code_prefix").isNull()) | (col("customer_city").isNull()) | (col("customer_state").isNull())
        )
)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, abfab595-ceba-4389-9981-bd8b63376862)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 46, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 18a7751f-dbdf-4201-abc2-a60bad94eb0c)

##### **Check for duplicates**

In [45]:
df1_duplicates_check = (
    df1.groupby(col("customer_id"))
    .count()
    .filter(col("count")>1)
)

display(df1_duplicates_check)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c72fb596-ad33-4f30-b76f-7ad9eae9407c)

In [46]:
df1_duplicates_check2 = (
    df1.groupby(col("customer_unique_id"))
    .count()
    .filter(col("count")>1)
)

display(df1_duplicates_check2)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 48, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 087901a4-a053-463f-b0de-52ec000d14f1)

##### **Check data consistency**

In [47]:
df1_consistency_check = (
    df1.select("customer_city").distinct()
)
display(df1_consistency_check)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 49, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7286f8a4-30c6-4ab1-88b3-3386df30ee74)

In [48]:
df1_consistency_check2 = (
    df1.select("customer_state").distinct()
)
display(df1_consistency_check2)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 263c0206-9637-4689-933a-8b3911c07aed)

##### **Data Validation**

In [49]:
df1_data_validation = (
    df1.alias("c").join(df10.alias("s"),col("c.customer_state") == col("s.state_code"),"left")
    .select(
        "c.*",
        when(col("s.state_code").isNull(),'Not Valid')
        .otherwise('Valid')
        .alias("state_valid_flag")
        )
)
check_state = (df1_data_validation.filter(col("state_valid_flag") == 'Not Valid'))
display(check_state)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 51, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 302c9fdc-2ff5-46d0-b60d-954aa78be92b)

In [50]:
df1_data_validation_city = (
    df1.alias("c1")
    .join(
        df11.alias("c2"),(col("c1.customer_state") == col("c2.state_code")) &
        (trim(col("c1.customer_city")) == lower((("c2.city_name")))),"left"
     )
    .select(
    "c1.*",
    when(col("c2.city_name").isNull(), "Not Valid")
    .otherwise("Valid")
    .alias("city_valid_flag")
    )
)

check_city = (df1_data_validation_city.filter(col("city_valid_flag") == 'Not Valid'))

display(check_city)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 52, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 67aafd03-6937-4a4d-90a5-ee245214d62c)

In [51]:
ad = df10.filter(col("state_name") == 'ceara')
display(ad)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 53, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ce2ff1a0-2717-486f-a911-7c090d14e41c)

In [52]:
def normalize_column(column):
    column = regexp_replace(column, "á|à|ã|â|ä", "a")
    column = regexp_replace(column, "é|è|ê|ë", "e")
    column = regexp_replace(column, "í|ì|î|ï", "i")
    column = regexp_replace(column, "ó|ò|õ|ô|ö", "o")
    column = regexp_replace(column, "ú|ù|û|ü", "u")
    column = regexp_replace(column, "ç", "c")

    column = lower(trim(column))
    column = regexp_replace(column, "[^a-z0-9 ]", "")
    column = regexp_replace(column, " +", " ")
    return column

df1_clean = (
    df1
    .withColumn("customer_state_clean", trim(upper(col("customer_state"))))
    .withColumn("customer_city_clean", normalize_column(col("customer_city")))
)

df11_clean = (
    df11
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("city_name_clean", normalize_column(col("city_name")))
)

df1_data_validation_city2 = (
    df1_clean.alias("c1")
    .join(
        df11_clean.alias("c2"),
        (col("c1.customer_state") == col("c2.state_code")) &
        (col("c1.customer_city_clean") == col("c2.city_name_clean")),
        "left"
    )
    .select(
        "c1.*",
        when(col("c2.city_name_clean").isNull(), "Not Valid")
        .otherwise("Valid")
        .alias("city_valid_flag")
    )
)

check_city2 = df1_data_validation_city2.filter(col("city_valid_flag") == "Not Valid")

display(check_city2)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 54, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c50fb69d-63bc-4fd7-b9fd-dfb1828c2156)

In [53]:
df11_clean.filter(col("city_name_clean").like("%arace%")).show()

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 55, Finished, Available, Finished, False)

+----------+----------+---------+--------------+---------------+
|state_code|state_name|city_name|city_ibge_code|city_name_clean|
+----------+----------+---------+--------------+---------------+
+----------+----------+---------+--------------+---------------+



In [54]:
total_customers = df1_clean.count()

matched = (
    df1_clean.alias("c1")
    .join(
        df11_clean.alias("c2"),
        (col("c1.customer_state") == col("c2.state_code")) &
        (col("c1.customer_city_clean") == col("c2.city_name_clean")),
        "inner"
    )
    .count()
)

print("Match %:", matched / total_customers * 100)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 56, Finished, Available, Finished, False)

Match %: 98.12887702163518


### **Final Table**

In [4]:
def normalize_column(column):
    column = regexp_replace(column, "á|à|ã|â|ä", "a")
    column = regexp_replace(column, "é|è|ê|ë", "e")
    column = regexp_replace(column, "í|ì|î|ï", "i")
    column = regexp_replace(column, "ó|ò|õ|ô|ö", "o")
    column = regexp_replace(column, "ú|ù|û|ü", "u")
    column = regexp_replace(column, "ç", "c")

    column = lower(trim(column))
    column = regexp_replace(column, "[^a-z0-9 ]", "")
    column = regexp_replace(column, " +", " ")
    return column

df1_clean = (
    df1
    .withColumn("customer_state_clean", trim(upper(col("customer_state"))))
    .withColumn("customer_city_clean", normalize_column(col("customer_city")))
)
df10_clean = (
    df10
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("state_name_clean", normalize_column(col("state_name")))
)
df11_clean = (
    df11
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("city_name_clean", normalize_column(col("city_name")))
)

ref_city = df11_clean.select("city_name_clean").distinct()

ref_state = df10_clean.select("state_code").distinct()

ref_state_city = df11_clean.select(
    "state_code",
    "city_name_clean"
).distinct()

df1_final = (
df1_clean.alias("c") \
.join(
    ref_city.alias("city_ref"),
    col("c.customer_city_clean") == col("city_ref.city_name_clean"),
    "left"
)
.join(
    ref_state.alias("state_ref"),
    col("c.customer_state_clean") == col("state_ref.state_code"),
    "left"
)
.join(
    ref_state_city.alias("sc_ref"),
    (col("c.customer_state_clean") == col("sc_ref.state_code")) &
    (col("c.customer_city_clean") == col("sc_ref.city_name_clean")),
    "left"
)
.withColumn(
    "is_city_valid",
    when(col("city_ref.city_name_clean").isNull(),0)
    .otherwise(1)
)
.withColumn(
    "is_state_valid",
    when(col("state_ref.state_code").isNull(),0)
    .otherwise(1)
)
.withColumn(
    "is_state_city_valid",
    when(col("sc_ref.state_code").isNull(),0)
    .otherwise(1)
)
.filter(col("c.customer_id").isNotNull())
.select(
    col("c.customer_id"),
    col("c.customer_unique_id"),
    col("c.customer_zip_code_prefix"),
    col("c.customer_city_clean").alias("customer_city"),
    col("c.customer_state_clean").alias("customer_state"),
    "is_city_valid",
    "is_state_valid",
    "is_state_city_valid"
    )
)

StatementMeta(, fe3f9746-2d10-40be-bea8-0b83ada7c71e, 6, Finished, Available, Finished, False)

In [56]:
display(df1_final)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 58, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b426fc4f-980b-4be1-b9dd-f03d1f82d383)

In [5]:
df1_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Olist_Silver_Lakehouse.dbo.silver_olist_customers")

StatementMeta(, fe3f9746-2d10-40be-bea8-0b83ada7c71e, 7, Finished, Available, Finished, False)

### **Table : olist_order_items**

In [58]:
display(df3)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9e642128-2f5e-4824-9bc3-055574111e89)

##### **Check For Nulls**

In [59]:
df3_nulls_check = (
    df3.filter(
        (col("order_id").isNull()) | 
        (col("order_item_id").isNull()) |
        (col("product_id").isNull()) |
        (col("seller_id").isNull()) |
        (col("product_id").isNull()) |
        (col("shipping_limit_date").isNull()) |
        (col("price").isNull()) |
        (col("freight_value").isNull()) 
        )
)

display(df3_nulls_check)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 61, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a150ec87-d293-40cc-8677-ea51244a5133)

In [60]:
df3.filter(col("price").isNull()).count()

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 62, Finished, Available, Finished, False)

1161

In [61]:
df3.count()

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 63, Finished, Available, Finished, False)

114861

In [62]:
df3_null_checking = (
    df3.join(df6,df3.order_id == df6.order_id,"left")
) 

df3_order_status = df3_null_checking.filter(col("price").isNull())

display(df3_order_status)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 64, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a2f62819-6878-42b5-a8f1-acd7456af1f5)

In [63]:
display(df6)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 65, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2066f9c7-5f80-4499-a0d5-e501015f603e)

In [64]:
display(df6.select("order_status").distinct())

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 66, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2c9f59e9-bc82-4c06-9a7c-2863d0b2b39e)

In [65]:
df3_price_null = df3.filter(col("price").isNull())

null_price_contribution = df3_price_null.alias("n").join(
    df6.alias("o"),
    col("n.order_id") == col("o.order_id"),
    "left"
) \
.groupBy("o.order_status") \
.count() \
.orderBy(col("count").desc())

display(null_price_contribution)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 67, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6c5d6ddd-6d0e-41ad-a26c-f8c623c6e37b)

##### **Data Validation**

In [66]:
df3_price_validation = (df3.alias("oi").join(df6.alias("o"),"order_id","left").withColumn(
    "price_status",
    when(col("oi.price").isNull(),"price_missing")
    .when((col("oi.price").isNull()) & (col("o.order_status") == "delivered"),"delivered_price_missing")
    .otherwise("Valid")
    )
.select("oi.*","price_status")
)

display(df3_price_validation)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 68, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 58e84b11-6e1d-4dcb-af2d-a2e82cb8c50c)

In [67]:
df3_negative_value_check = (
    df3.filter(col("price") < 0)
)

display(df3_negative_value_check)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 69, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 962e64dc-ff7a-4d3b-b6e0-908057cdee96)

In [68]:
df3_negative_value_check = (df3.filter(col("price")<0))

df3_neg_validation = (
df3_negative_value_check.alias("n")
.join(df6.alias("o"),"order_id","left")
.filter(col("o.order_status") == "delivered")
.select("n.*")
)

display(df3_neg_validation)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 70, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 97f0ebf6-28ac-44d4-b218-b5a589811dea)

In [69]:
df3_neg_solution = (
    df3.alias("oi")
    .join(df6.alias("o"),"order_id","left")
    .withColumn(
        "is_refunded",
        when(col("price") < 0, "Yes")
        .otherwise("No")
    )
    .select("oi.*","is_refunded")
)

display(df3_neg_solution)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 71, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ab243ce0-6684-4f75-8a9b-1f3d6664cb76)

In [70]:
df3_neg_freight = (
    df3.filter(col("freight_value") < 0)
)

display(df3_neg_freight)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 72, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dab38458-621c-4130-ad38-5bcf374c2d6e)

##### **Referential Integrity Check**

In [71]:
df3_ref_check = (
    df3.alias("oi").join(
        df4.alias("po"),
        "order_id",
        "left"
    )
    .filter(col("po.order_id").isNull())
)

display(df3_ref_check)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 73, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 59d3606d-1e17-41fb-9062-9370d669b074)

In [72]:
df3_wwe = (
    df3.filter(col("order_id") == '1a57108394169c0b47d8f876acc9ba2d')
)

display(df3_wwe)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 74, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8648c15b-5d92-4312-9bdd-2c98d56d9baf)

In [73]:
df4_wwe = (
    df4.filter(col("order_id") == '744bade1fcf9ff3f31d860ace076d422')
)

display(df4_wwe)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 75, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f4d3ec5c-4f29-4173-86bf-8bda11801a32)

### **Final Table**

In [9]:
df_silver_orders = spark.read.table("Olist_Silver_Lakehouse.dbo.silver_olist_orders")

df3_final = (
    df3.alias("oi").withColumn(
        "ranking",
        row_number().over(Window.partitionBy("order_id","order_item_id")
        .orderBy(col("ingestion_timestamp").desc()))
    )
    .filter(col("ranking") == 1)
    .drop("ranking")
    .join(df_silver_orders.alias("o"),col("o.order_id") == col("oi.order_id"),"left")
    .withColumn(
    "is_delivered_price_missing",
    when((col("oi.price").isNull()) & (col("o.order_status") == "delivered"),1)
    .otherwise(0)
    )
    .withColumn(
    "is_price_missing",
    when(col("oi.price").isNull(),1)
    .otherwise(0)
    )
    .withColumn(
    "is_price_valid",
    when((col("oi.price") >= 0) & (col("oi.price").isNotNull()),1)
    .otherwise(0)
    )
    .withColumn(
        "is_refunded",
        when(col("oi.price") < 0, 1)
        .otherwise(0)
    )
     .withColumn(
        "is_delivered_freight_value_missing",
        when((col("oi.freight_value").isNull()) & (col("o.order_status") == "delivered" ),1)
        .otherwise(0)
    )
    .withColumn(
        "is_freight_value_missing",
        when(col("oi.freight_value").isNull(),1)
        .otherwise(0)
    )
    .withColumn(
        "is_freight_value_valid",
        when((col("oi.freight_value") >= 0) & (col("oi.freight_value").isNotNull()),1)
        .otherwise(0)
    )
    .select("oi.*","is_price_missing","is_price_valid","is_delivered_price_missing","is_delivered_freight_value_missing","is_freight_value_missing","is_freight_value_valid","is_refunded")
)


StatementMeta(, 94260921-8355-492b-9249-0421733c78fb, 11, Finished, Available, Finished, False)

In [10]:
df_silver_order_items = (
    df3_final.select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "is_price_missing",
        "is_price_valid",
        "is_delivered_price_missing",
        "freight_value",
        "is_delivered_freight_value_missing",
        "is_freight_value_missing",
        "is_freight_value_valid",
        "is_refunded",
        "ingestion_timestamp"
    )
)

StatementMeta(, 94260921-8355-492b-9249-0421733c78fb, 12, Finished, Available, Finished, False)

In [12]:
df_silver_order_items.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Olist_Silver_Lakehouse.dbo.silver_olist_order_items")

StatementMeta(, 94260921-8355-492b-9249-0421733c78fb, 14, Finished, Available, Finished, False)

### **Table : olist_orders**

In [77]:
display(df6)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 79, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0b23b537-6a2f-457c-a88f-dea71fd9ebd4)

##### **Check for nulls**

In [78]:
df6_null_check1 = (
    df6.filter(col("order_id").isNull())
)

df6_null_check2 = (
    df6.filter(col("customer_id").isNull())
)

display(df6_null_check2)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 80, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1ac80e1c-bfc6-46ca-b6d7-324ce2927961)

##### **Check for duplicates**

In [79]:
df6_check_duplicatess = (
    df6.groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

df6_check_duplicates = (df6_check_duplicatess.alias("d") \
.join(df6.alias("o"),"order_id","left") \
.select("d.order_id","d.count","o.order_purchase_timestamp","order_delivered_carrier_date","order_delivered_customer_date")
)


display(df6_check_duplicates)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 81, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, aef95938-3066-4d56-b234-4d05cf104e66)

In [80]:
df6_duplicates_sol = (
    df6.withColumn(
        "ranking",
        row_number().over(Window.partitionBy("order_id").orderBy(desc("order_delivered_customer_date")))
    )
    .filter(col("ranking") == 1)
    .drop("ranking")
)

display(df6_duplicates_sol)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 82, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 880dce5b-4717-484c-8d05-875fc423a261)

##### **Consistency check**

In [81]:
df6_consistency_check = (
    df6.select("order_status").distinct()
)

display(df6_consistency_check)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 83, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a9118578-ccaa-4a88-b9d9-957a7588c99c)

##### **Date Validation**

In [82]:
df6_date_validation1 = (
    df6.filter(col("order_purchase_timestamp") > col("order_approved_at"))
)

df6_date_validation2 = (
    df6.filter(col("order_approved_at") > col("order_delivered_carrier_date"))
)

df6_date_validation3 = (
    df6.filter(col("order_delivered_carrier_date") > col("order_delivered_customer_date"))
)

df6_date_validation4 = (
    df6.filter(col("order_delivered_customer_date") > col("order_estimated_delivery_date"))
)

display(df6_date_validation1)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 84, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f40c1b12-76bf-42df-9766-4a5b4c5435e4)

In [83]:
df6_date_flags = (
    df6.withColumn(
        "is_purchase_after_approved",
        when(
            (col("order_purchase_timestamp").isNotNull()) &
            (col("order_approved_at").isNotNull()) &
            (col("order_purchase_timestamp") > col("order_approved_at")),
            1
            )
        .otherwise(0)
    )
    .withColumn(
        "is_approval_after_carrier",
        when(
            (col("order_delivered_carrier_date").isNotNull()) &
            (col("order_approved_at").isNotNull()) &
            (col("order_approved_at") > col("order_delivered_carrier_date")),
            1
            )
        .otherwise(0)
    )
    .withColumn(
        "is_carrier_after_customer",
        when(
            (col("order_delivered_carrier_date").isNotNull()) &
            (col("order_delivered_customer_date").isNotNull()) &
            (col("order_delivered_carrier_date") > col("order_delivered_customer_date")),
            1
            )
        .otherwise(0)
    )
    .withColumn(
        "is_late_delivery",
        when(
            (col("order_delivered_carrier_date").isNotNull()) &
            (col("order_estimated_delivery_date").isNotNull()) &
            (col("order_delivered_customer_date") > col("order_estimated_delivery_date")),
            1
            )
        .otherwise(0)
    )
)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 85, Finished, Available, Finished, False)

In [84]:
display(df6_date_flags)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 86, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8bbf1531-2bc3-49a3-8761-2d0557403b77)

### **Final Table**

In [85]:
df6_final = (
        df6.withColumn(
        "ranking",
        row_number().over(Window.partitionBy("order_id")
        .orderBy(desc("order_delivered_customer_date"),desc("order_approved_at"),desc("order_purchase_timestamp")))
    )
    .filter(col("ranking") == 1)
    .drop("ranking")

    .withColumn(
        "is_purchase_after_approved",
        when(
            (col("order_purchase_timestamp").isNotNull()) &
            (col("order_approved_at").isNotNull()) &
            (col("order_purchase_timestamp") > col("order_approved_at")),
            1
            )
        .otherwise(0)
    )
    .withColumn(
        "is_approval_after_carrier",
        when(
            (col("order_delivered_carrier_date").isNotNull()) &
            (col("order_approved_at").isNotNull()) &
            (col("order_approved_at") > col("order_delivered_carrier_date")),
            1
            )
        .otherwise(0)
    )
    .withColumn(
        "is_carrier_after_customer",
        when(
            (col("order_delivered_carrier_date").isNotNull()) &
            (col("order_delivered_customer_date").isNotNull()) &
            (col("order_delivered_carrier_date") > col("order_delivered_customer_date")),
            1
            )
        .otherwise(0)
    )
    .withColumn(
        "is_late_delivery",
        when(
            (col("order_status") == "delivered") &
            (col("order_delivered_customer_date").isNotNull()) &
            (col("order_estimated_delivery_date").isNotNull()) &
            (col("order_delivered_customer_date") > col("order_estimated_delivery_date")),
            1
            )
        .otherwise(0)
    )
    .select("order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "is_purchase_after_approved","is_approval_after_carrier","is_carrier_after_customer","is_late_delivery")
)


StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 87, Finished, Available, Finished, False)

In [86]:
display(df6_final)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 88, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 003b9830-3da6-4fbf-95f9-a3bfd31dc0ac)

In [87]:
df6_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Olist_Silver_Lakehouse.dbo.silver_olist_orders")

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 89, Finished, Available, Finished, False)

### **Table : Olist_order_payments**

In [88]:
display(df4)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 90, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 106eb038-920a-42bc-a012-a68575564054)

##### **Check for nulls**

In [89]:
df4_null_check1 = (
    df4.filter((col("order_id").isNull()) |
    (col("payment_sequential").isNull()) |
    (col("payment_type").isNull()) |
    (col("payment_installments").isNull()) |
    (col("payment_value").isNull())
    )
)

display(df4_null_check1)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 91, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f4bccaeb-8d40-4c99-a438-f349faea9b2e)

##### **Check for duplicates**

In [90]:
df4_duplicates_check = (
    df4.groupBy("order_id" , "payment_sequential")
    .count()
    .filter(col("count") > 1)
)

display(df4_duplicates_check)


StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 92, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0751918a-5e3f-431a-8f3f-cfcf73eac9e9)

##### **Check for consistency**

In [91]:
df4_consistency_check1 = (
    df4.select("payment_sequential").distinct()
)

display(df4_consistency_check1)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 93, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d828808c-c3f4-4861-a2af-3071471a6c23)

In [92]:
df4_consistency_check2 = (
    df4.select("payment_type").distinct()
)

display(df4_consistency_check2)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 94, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f94c6834-2542-4c93-90bc-2a0935d6504c)

In [93]:
df4_consistency_check3 = (
    df4.groupBy("order_id") \
    .agg(max("payment_sequential").alias("max_seq"),
    count("*").alias("records"))
    .filter(col("max_seq") != col("records"))
)
display(df4_consistency_check3)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 95, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 134dfa33-007c-41dd-a20c-1aa0da775e6a)

##### **Check Referential Integrity**

In [94]:
df4_referential_integrity_check = (
    df4.join(
        df6, "order_id" , "left_anti"
    )
)

display(df4_referential_integrity_check)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 96, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7d730f8f-089e-49f9-a29d-05d47efa79dd)

##### **Check for negative values**

In [95]:
df4_neg_val_check = (
    df4.filter(col("payment_value") < 0)
)

display(df4_neg_val_check)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 97, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d6b6d098-e9d6-4be3-b090-41d960848b1c)

In [96]:
we = (df4.groupBy("order_id") \
    .agg(max("payment_sequential").alias("max_seq"),
         count("*").alias("records"))
)
display(we)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 98, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 61ea683a-2256-4dc1-b0cf-bb04a6c12e31)

##### **Value Validation**

In [97]:
total_revenue_payment = (
    df4.agg(sum("payment_value"))
)

display(total_revenue_payment)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 99, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a0f18b51-5314-44c8-a8c7-743cf2a5ba60)

In [102]:
df = spark.read.table("Olist_Silver_Lakehouse.dbo.silver_olist_order_items")
total_revenue_of_order_item = (
    df.withColumn("total_revenue", col("price") + col("freight_value"))
    .filter((col("is_price_valid") == 1) & (col("is_freight_value_valid") == 1))
)

tt = (
    total_revenue_of_order_item.select(sum("total_revenue"))
)
display(tt)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 104, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0506f63d-20a2-40b1-af58-ccec822941ad)

In [103]:
payments_per_order = (df4.groupBy("order_id") \
    .agg(sum("payment_value").alias("total_payment")))

display(payments_per_order)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 105, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fa4936bc-4a18-4496-8fa6-4f7b3ceeac38)

### **Final Table**

In [ ]:
valid_payment_types = ["credit_card", "boleto", "voucher", "debit_card"]
df4_final =(
    df4.filter(
        (col("payment_sequential") >= 1) &
        (col("payment_type").isin(valid_payment_types)) &
        (col("payment_installments") >= 1) &
        (col("payment_value") >= 0)
        )
    .select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value",
        "ingestion_timestamp"
    )
)

In [ ]:
df4_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Olist_Silver_Lakehouse.dbo.silver_olist_order_payments")

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, -1, Cancelled, , Cancelled, True)

### **Table : Olist_order_reviews**

In [ ]:
display(df5)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, -1, Cancelled, , Cancelled, True)

#### **Check for nulls**

In [ ]:
df5_null_check = (
    df5.filter((col("review_id").isNull()) | (col("order_id").isNull()) | (col("review_score").isNull()))
)

display(df5_null_check)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, -1, Cancelled, , Cancelled, True)

In [ ]:
df5_null_check2 = (
    df5.filter((col("review_creation_date").isNull()) | (col("review_answer_timestamp").isNull()))
)

display(df5_null_check2)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, -1, Cancelled, , Cancelled, True)

##### **Consistency Check**

In [ ]:
df5_consistency_check = (
    df5.select("review_score").distinct()
)

display(df5_consistency_check)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, -1, Cancelled, , Cancelled, True)

##### **Date Validation**

In [ ]:
df5_date_check = (
    df5.filter(col("review_creation_date") > col("review_answer_timestamp"))
)

display(df5_date_check)

##### **Negative value check**

In [19]:
df5_neg_val_check = (
    df5.filter(col("review_score") < 0)
)

display(df5_neg_val_check)

StatementMeta(, a2bd06e7-f054-44ac-99a2-029379a3048e, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a6057548-89b5-41b6-a809-e1ad5227735b)

### **Final Table**

In [17]:
df5_final = (
    df5.filter(
        (col("review_id").isNotNull()) &
        (col("order_id").isNotNull()) &
        (col("review_creation_date").isNotNull())
    )
    .withColumn(
        "is_review_score_valid",
        when(col("review_score").between(1,5),1)
        .otherwise(0)
    )
    .withColumn(
        "is_review_with_comment",
        when((col("review_comment_title").isNotNull()) | (col("review_comment_message").isNotNull()),1)
        .otherwise(0)
    )
    .withColumn(
        "is_answer_after_review",
        when((col("review_answer_timestamp").isNotNull()) & (col("review_answer_timestamp") >= col("review_creation_date")),1)
        .otherwise(0)
    )
    .select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp",
        "is_review_score_valid",
        "is_review_with_comment",
        "is_answer_after_review",
        "ingestion_timestamp"
    )
)

StatementMeta(, dfd98a7e-6f14-427a-8105-0475404ec975, 19, Finished, Available, Finished, False)

In [24]:
df5_final.write.mode("overwrite").saveAsTable("Olist_Silver_Lakehouse.dbo.silver_olist_order_reviews")

StatementMeta(, dfd98a7e-6f14-427a-8105-0475404ec975, 26, Finished, Available, Finished, False)

### **Table : olist_products**

In [ ]:
display(df7)

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] This spark job can't be run because you have hit a spark compute or API rate limit. To run this spark job, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. HTTP status code: 430 {Learn more} HTTP status code: 430.

##### **Check for nulls**

In [30]:
df7_null_check1 = (
    df7.filter(col("product_id").isNull())
)

display(df7_null_check1)

StatementMeta(, a2bd06e7-f054-44ac-99a2-029379a3048e, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 50dee686-a27f-4a3f-abfe-eec6d0a7646d)

In [29]:
df7_null_check2 = (
    df7.filter(col("product_category_name").isNull())
)

display(df7_null_check2)

StatementMeta(, a2bd06e7-f054-44ac-99a2-029379a3048e, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 09e2a5ca-1755-4312-a100-b4f0437f2641)

### **Final Table**

In [19]:
df7_final = (
    df7.filter(
        col("product_id").isNotNull()
    )
    .alias("p").join(df9.alias("pc"), "product_category_name", "left")
    .withColumn(
        "is_product_category_missing",
        when(col("p.product_category_name").isNull(),1)
        .otherwise(0)
    )
    .withColumn(
    "is_product_name_length_valid",
    when(col("p.product_name_lenght") >= 0,1)
    .otherwise(0)
    )
    .withColumn(
        "is_product_weight_valid",
        when(col("p.product_weight_g") >= 0,1)
        .otherwise(0)
    )
    .withColumn(
        "is_product_photos_valid",
        when(col("p.product_photos_qty") >= 0,1)
        .otherwise(0)
    )
    .withColumn(
        "is_product_dimension_valid",
        when((col("p.product_length_cm") >= 0) & (col("p.product_height_cm") >= 0) & (col("p.product_width_cm") >= 0),1)
        .otherwise(0)
    )
    .withColumn(
        "is_category_translation_available",
        when(col("pc.product_category_name").isNull(),0)
        .otherwise(1)
    )
    .select(
        col("p.product_id"),
        col("p.product_category_name"),
        col("p.product_name_lenght"),
        col("p.product_description_lenght"),
        col("p.product_photos_qty"),
        col("p.product_weight_g"),
        col("p.product_length_cm"),
        col("p.product_height_cm"),
        col("p.product_width_cm"),
        "is_product_category_missing",
        "is_category_translation_available",
        "is_product_name_length_valid",
        "is_product_weight_valid",
        "is_product_photos_valid",
        "is_product_dimension_valid",
        col("p.ingestion_timestamp")
    )
)

StatementMeta(, dfd98a7e-6f14-427a-8105-0475404ec975, 21, Finished, Available, Finished, False)

In [23]:
df7_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Olist_Silver_Lakehouse.dbo.silver_olist_products")

StatementMeta(, dfd98a7e-6f14-427a-8105-0475404ec975, 25, Finished, Available, Finished, False)

### **Table : olist_sellers**

In [35]:
display(df8)

StatementMeta(, a2bd06e7-f054-44ac-99a2-029379a3048e, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 43a88c0e-1900-4fc1-98a8-9bbdc98411d6)

##### **Check For Nulls**

In [37]:
df8_null_check1 = (
    df8.filter(col("seller_id").isNull())
)

display(df8_null_check1)

StatementMeta(, a2bd06e7-f054-44ac-99a2-029379a3048e, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c9e1d9da-8c3a-4e1e-87dc-a3c8ea3114b2)

In [40]:
df8_null_check2 = (
    df8.filter((col("seller_zip_code_prefix").isNull()) | (col("seller_city").isNull()) | (col("seller_state").isNull()))
)

display(df8_null_check2)

StatementMeta(, a2bd06e7-f054-44ac-99a2-029379a3048e, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 108a51b8-2914-4513-b119-6a13d0f94c11)

##### **Check For Duplicates**

In [43]:
df8_duplicates_check = (
    df8.groupby("seller_id","seller_zip_code_prefix") 
    .count() \
    .filter(col("count") > 1)
)

display(df8_duplicates_check)

StatementMeta(, a2bd06e7-f054-44ac-99a2-029379a3048e, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7d49c8a8-5a87-4386-813f-b3108e7e3393)

In [16]:
def normalize_column(column):
    
    column = regexp_replace(column, "á|à|ã|â|ä", "a")
    column = regexp_replace(column, "é|è|ê|ë", "e")
    column = regexp_replace(column, "í|ì|î|ï", "i")
    column = regexp_replace(column, "ó|ò|õ|ô|ö", "o")
    column = regexp_replace(column, "ú|ù|û|ü", "u")
    column = regexp_replace(column, "ç", "c")
    column = regexp_replace(column, "[-/].*", "")

    column = lower(trim(column))
    column = regexp_replace(column, "[^a-z0-9 ]", "")
    column = regexp_replace(column, " +", " ")
    return column

df8_clean = (
    df8
    .withColumn("seller_state_clean", trim(upper(col("seller_state"))))
    .withColumn("seller_city_clean", normalize_column(col("seller_city")))
)

df11_clean = (
    df11
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("city_name_clean", normalize_column(col("city_name")))
)


df8_city_check = (
    df8_clean.alias("s").join(
        df11_clean.alias("c"), 
        ((col("c.city_name_clean")) == (col("s.seller_city_clean"))) &
        (col("c.state_code") == col("s.seller_state_clean")),
        "left"
        )
    .filter(col("c.city_name_clean").isNull())
)

display(df8_city_check)


StatementMeta(, 0412f363-cff1-435f-bb68-92a709ffb2a4, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e18aa126-dbed-4f10-ae8b-f62a9bb0cf8a)

In [8]:
df8_state_check = (
    df8.alias("s").join(df11.alias("c"), trim(lower(col("c.state_code"))) == trim(lower(col("s.seller_state"))) , "left")
    .filter(col("c.state_code").isNull())
)

display(df8_state_check)

StatementMeta(, a3ae54dd-90a9-4900-84e7-75ccaf3104c4, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 96d0a7c7-1653-4894-a7fb-76d094e7c75e)

In [56]:
cd = df11.filter(col("state_name") == "GO")

display(cd)

StatementMeta(, a2bd06e7-f054-44ac-99a2-029379a3048e, 58, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0ce8031c-898b-483a-852a-f5d4fa012e97)

In [57]:
display(df11)

StatementMeta(, a2bd06e7-f054-44ac-99a2-029379a3048e, 61, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a0621f24-e2ad-4208-aa04-a3105f453ed3)

### **Final Table**

In [21]:
def normalize_column(column):
    
    column = regexp_replace(column, "á|à|ã|â|ä", "a")
    column = regexp_replace(column, "é|è|ê|ë", "e")
    column = regexp_replace(column, "í|ì|î|ï", "i")
    column = regexp_replace(column, "ó|ò|õ|ô|ö", "o")
    column = regexp_replace(column, "ú|ù|û|ü", "u")
    column = regexp_replace(column, "ç", "c")
    column = regexp_replace(column, "[-/].*", "")

    column = lower(trim(column))
    column = regexp_replace(column, "[^a-z0-9 ]", "")
    column = regexp_replace(column, " +", " ")
    return column

df8_clean = (
    df8
    .withColumn("seller_state_clean", trim(upper(col("seller_state"))))
    .withColumn("seller_city_clean", normalize_column(col("seller_city")))
)

df10_clean = (
    df10
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("state_name_clean", normalize_column(col("state_name")))
)

df11_clean = (
    df11
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("city_name_clean", normalize_column(col("city_name")))
)

ref_city = df11_clean.select(col("city_name_clean")).distinct()

ref_state = df10_clean.select(col("state_code")).distinct()

ref_state_city = df11_clean.select("state_code", "city_name_clean").distinct()

df8_final = (
    df8_clean.alias("s")
    .join(
        ref_city.alias("city_ref"),
        col("city_ref.city_name_clean") == col("s.seller_city_clean"),
        "left"
    )
    .join(
        ref_state.alias("state_ref"),
        col("state_ref.state_code") == col("s.seller_state_clean"),
        "left"
    )
    .join(
        ref_state_city.alias("sc_ref"),
       (col("s.seller_state_clean") == col("sc_ref.state_code")) &
       (col("s.seller_city_clean") == col("sc_ref.city_name_clean")),
       "left"
    )
    .withColumn(
        "is_city_valid",
        when(col("city_ref.city_name_clean").isNull(),0)
        .otherwise(1)
    )
    .withColumn(
        "is_state_valid",
        when(col("state_ref.state_code").isNull(),0)
        .otherwise(1)
    )
    .withColumn(
        "is_state_city_valid",
        when(col("sc_ref.state_code").isNull(),0)
        .otherwise(1)
    )
    .select(
        col("s.seller_id"),
        col("s.seller_zip_code_prefix"),
        col("s.seller_city"),
        col("s.seller_state"),
        "is_city_valid",
        "is_state_valid",
        "is_state_city_valid",
        col("s.ingestion_timestamp")
    )
)

StatementMeta(, dfd98a7e-6f14-427a-8105-0475404ec975, 23, Finished, Available, Finished, False)

In [34]:
display(df8_final)

StatementMeta(, 0412f363-cff1-435f-bb68-92a709ffb2a4, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8b8188fb-3789-47af-a3e6-4063199812c5)

In [22]:
df8_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Olist_Silver_Lakehouse.dbo.silver_olist_sellers")

StatementMeta(, dfd98a7e-6f14-427a-8105-0475404ec975, 24, Finished, Available, Finished, False)

### **Table : olist_product_category_translation**

In [38]:
display(df9)

StatementMeta(, 0412f363-cff1-435f-bb68-92a709ffb2a4, 40, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 07e2f8b8-99e3-4639-9054-0ded3f03dac0)

##### **Check For Nulls**

In [40]:
df9_null_check = (
    df9.filter((col("product_category_name").isNull()) | (col("product_category_name_english").isNull()))
)

display(df9_null_check)

StatementMeta(, 0412f363-cff1-435f-bb68-92a709ffb2a4, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 54e746ca-e2d4-4a6a-ba62-55d5b8128bfa)

##### **Check For Duplicates**

In [42]:
df9_duplicates_check1 = (
    df9.groupBy("product_category_name")
    .count() \
    .filter(col("count") > 1)
)

display(df9_duplicates_check1)

StatementMeta(, 0412f363-cff1-435f-bb68-92a709ffb2a4, 44, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eb9167af-ee83-4435-8f22-0411be84a883)

In [43]:
df9_duplicates_check2 = (
    df9.groupBy("product_category_name_english")
    .count() \
    .filter(col("count") > 1)
)

display(df9_duplicates_check2)

StatementMeta(, 0412f363-cff1-435f-bb68-92a709ffb2a4, 45, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6fd13d5f-49cf-47e2-b80e-4aac507cf0a8)

In [107]:
display(df7.select("product_category_name").distinct())

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 109, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b6227db4-b10c-4c22-bbaa-c13f1d393c01)

In [108]:
display(df9.select("product_category_name").distinct())

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 110, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 16ed01f4-0789-4148-8e68-a60e1bdf2129)

In [109]:
silver_product = spark.read.table("Olist_Silver_Lakehouse.silver_olist_products")
check = (
    silver_product.alias("p").join(
    df9.alias("pc"),
    "product_category_name",
    "left_anti"
)
.filter(col("p.product_category_name").isNotNull())
)

display(check)

StatementMeta(, d57a0b54-0299-48e8-9501-56be20eaef0a, 111, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c635d842-44e7-4dec-9ddf-eed74230a7a9)

In [64]:
check2 = (
    silver_product
    .withColumn("cat_clean", trim(lower(col("product_category_name"))))
    .join(
        df9.withColumn("cat_clean", trim(lower(col("product_category_name")))),
        "cat_clean",
        "left_anti"
    )
    .filter(col("cat_clean").isNotNull())
)

display(check2)

StatementMeta(, 0412f363-cff1-435f-bb68-92a709ffb2a4, 66, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 11a25d73-dd1e-405c-bbcc-5fa94519bbe2)

In [65]:
df9.filter(
    col("product_category_name").isin(
        "pc_gamer",
        "portateis_cozinha_e_preparadores_de_alimentos"
    )
).show(truncate=False)

StatementMeta(, 0412f363-cff1-435f-bb68-92a709ffb2a4, 67, Finished, Available, Finished, False)

+---------------------+-----------------------------+-------------------+
|product_category_name|product_category_name_english|ingestion_timestamp|
+---------------------+-----------------------------+-------------------+
+---------------------+-----------------------------+-------------------+



In [67]:
silver_product.filter(
    col("product_category_name").isin(
        "pc_gamer",
        "portateis_cozinha_e_preparadores_de_alimentos"
    )
).groupBy("product_category_name").count().show()

StatementMeta(, 0412f363-cff1-435f-bb68-92a709ffb2a4, 69, Finished, Available, Finished, False)

+---------------------+-----+
|product_category_name|count|
+---------------------+-----+
|             pc_gamer|    3|
| portateis_cozinha...|   10|
+---------------------+-----+



### **Final Table**

In [27]:
df9_final = (
    df9.filter(col("product_category_name").isNotNull() & col("product_category_name_english").isNotNull())
    .select(
        trim(lower(col("product_category_name"))).alias("product_category_name"),
        trim(lower(col("product_category_name_english"))).alias("product_category_name_english"),
        col("ingestion_timestamp")
    )
)

StatementMeta(, dfd98a7e-6f14-427a-8105-0475404ec975, 29, Finished, Available, Finished, False)

In [28]:
df9_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Olist_Silver_Lakehouse.dbo.silver_olist_product_category_translation")

StatementMeta(, dfd98a7e-6f14-427a-8105-0475404ec975, 30, Finished, Available, Finished, False)

### **Table : olist_geolocation**

In [10]:
display(df2)

StatementMeta(, cd4849f9-cfe4-4999-9294-de55d838a844, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e90ec93a-2ad6-442e-bbe1-19051abb366c)

##### **Check For Nulls**

In [11]:
df2_null_check = (
    df2.filter(col("geolocation_zip_code_prefix").isNull())
)

display(df2_null_check)

StatementMeta(, cd4849f9-cfe4-4999-9294-de55d838a844, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f7ad5497-37bd-417b-b8e8-246a8930d226)

In [12]:
df2_null_check2 = (
    df2.filter((col("geolocation_lat").isNull()) | (col("geolocation_lng").isNull()))
)

display(df2_null_check2)

StatementMeta(, cd4849f9-cfe4-4999-9294-de55d838a844, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 513e9c4b-68a5-451c-bc21-2cc785862c6f)

In [13]:
df2_null_check3 = (
    df2.filter((col("geolocation_city").isNull()) | (col("geolocation_state").isNull()))
)

display(df2_null_check3)


StatementMeta(, cd4849f9-cfe4-4999-9294-de55d838a844, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0575da8f-74a4-49cb-ade0-190a4d3f6bf1)

##### **Check For Duplicates**

In [16]:
df2_duplicates_check = (
    df2.groupBy("geolocation_zip_code_prefix")
    .count()
    .filter(col("count") > 1)
)

display(df2_duplicates_check)

StatementMeta(, cd4849f9-cfe4-4999-9294-de55d838a844, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4884f0b4-a4d4-4153-9d93-eb1edcccd7ee)

##### **City Column Validation**

In [20]:
def normalize_column(column):
    
    column = regexp_replace(column, "á|à|ã|â|ä", "a")
    column = regexp_replace(column, "é|è|ê|ë", "e")
    column = regexp_replace(column, "í|ì|î|ï", "i")
    column = regexp_replace(column, "ó|ò|õ|ô|ö", "o")
    column = regexp_replace(column, "ú|ù|û|ü", "u")
    column = regexp_replace(column, "ç", "c")
    column = regexp_replace(column, "[-/].*", "")

    column = lower(trim(column))
    column = regexp_replace(column, "[^a-z0-9 ]", "")
    column = regexp_replace(column, " +", " ")
    return column

df2_clean = (
    df2
    .withColumn("geolocation_state_clean", trim(upper(col("geolocation_state"))))
    .withColumn("geolocation_city_clean", normalize_column(col("geolocation_city")))
)

df11_clean = (
    df11
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("city_name_clean", normalize_column(col("city_name")))
)

city_validation = (
    df2_clean.select("geolocation_city_clean", "geolocation_state_clean").distinct()
    .alias("g").join(
    df11_clean.alias("r"),
    col("g.geolocation_city_clean") == col("r.city_name_clean"),
    "left_anti"
    )
)

display(city_validation)

StatementMeta(, cd4849f9-cfe4-4999-9294-de55d838a844, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e0eba01a-f75c-44dd-877c-663d6d9a7d11)

##### **State Column Validation**

In [ ]:
df11_state = (
    df11
    .withColumn("state_code", trim(upper(col("state_code"))))
)
            

df2_state = (
    df2
    .withColumn("geolocation_state", trim(upper(col("geolocation_state"))))
)

compare = (
    df2_state.select("geolocation_state").distinct()
    .alias("g")
    .join(
        df11_state.alias("r"),
        col("g.geolocation_state") == col("r.state_code"),
        "left"
        )
    .filter(col("r.state_code").isNull())
)

display(compare)

### **Final Table**

In [3]:
def normalize_column(column):
    column = regexp_replace(column, "á|à|ã|â|ä", "a")
    column = regexp_replace(column, "é|è|ê|ë", "e")
    column = regexp_replace(column, "í|ì|î|ï", "i")
    column = regexp_replace(column, "ó|ò|õ|ô|ö", "o")
    column = regexp_replace(column, "ú|ù|û|ü", "u")
    column = regexp_replace(column, "ç", "c")
    column = regexp_replace(column, "[-/].*", "")
    column = lower(trim(column))
    column = regexp_replace(column, "[^a-z0-9 ]", "")
    column = regexp_replace(column, " +", " ")
    return column

df2_clean = (
    df2
    .withColumn("geolocation_state_clean", trim(upper(col("geolocation_state"))))
    .withColumn("geolocation_city_clean", normalize_column(col("geolocation_city")))
)

df11_clean = (
    df11
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("city_name_clean", normalize_column(col("city_name")))
)

ref_city = df11_clean.select("city_name_clean").distinct()

ref_state = df11_clean.select("state_code").distinct()  

ref_state_city = df11_clean.select(
    "state_code",
    "city_name_clean"
).distinct()

df2_final = (
    df2_clean.alias("g")
    .join(
        ref_city.alias("city_ref"),
        col("g.geolocation_city_clean") == col("city_ref.city_name_clean"),  
        "left"
    )
    .join(
        ref_state.alias("state_ref"),
        col("g.geolocation_state_clean") == col("state_ref.state_code"),  
        "left"
    )
    .join(
        ref_state_city.alias("sc_ref"),
        (col("g.geolocation_state_clean") == col("sc_ref.state_code")) &  
        (col("g.geolocation_city_clean") == col("sc_ref.city_name_clean")),  
        "left"
    )
    .withColumn(
        "is_city_valid",
        when(col("city_ref.city_name_clean").isNull(), 0).otherwise(1)
    )
    .withColumn(
        "is_state_valid",
        when(col("state_ref.state_code").isNull(), 0).otherwise(1)
    )
    .withColumn(
        "is_state_city_valid",
        when(col("sc_ref.state_code").isNull(), 0).otherwise(1)
    )
    .select(
        col("g.geolocation_zip_code_prefix"),
        col("g.geolocation_lat"),
        col("g.geolocation_lng"),
        col("g.geolocation_city"),
        col("g.geolocation_state"),
        col("is_city_valid"),          
        col("is_state_valid"),         
        col("is_state_city_valid"),    
        col("g.ingestion_timestamp")
    )
)

StatementMeta(, 63117b16-cd16-4edd-9a29-cfa150052a73, 5, Finished, Available, Finished, False)

****

In [4]:
df2_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("Olist_Silver_Lakehouse.dbo.silver_olist_geolocation")

StatementMeta(, 63117b16-cd16-4edd-9a29-cfa150052a73, 6, Finished, Available, Finished, False)

In [3]:
%%sql
TRUNCATE TABLE dbo.olist_orders;
TRUNCATE TABLE dbo.olist_order_items;
TRUNCATE TABLE dbo.olist_order_payments;
TRUNCATE TABLE dbo.olist_order_reviews;
TRUNCATE TABLE dbo.olist_customers;
TRUNCATE TABLE dbo.olist_products;
TRUNCATE TABLE dbo.olist_sellers;
TRUNCATE TABLE dbo.olist_geolocation;
TRUNCATE TABLE dbo.product_category_name_translation;

StatementMeta(, 59219568-97ea-4687-8c53-1e60d308d94b, 13, Finished, Available, Finished, True)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>